# Semana 27 — A/B Tests, Statistical Power e Spillover em Redes
**Fase 6: Causal Inference e Experimentation** · Gabarito (notebook resolvido)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/applied-ai-architecture/blob/main/06-Phase-6-Causal-Inference/notebooks/semana_27_gabarito.ipynb)

Leitura de conceito desta semana: [Semana-27-Conceitos.md](https://github.com/NetoRibeiro/applied-ai-architecture/blob/main/06-Phase-6-Causal-Inference/docs/Semana-27-Conceitos.md)

Use este gabarito **depois** de tentar o notebook de exercícios. Compare a sua solução, não copie.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest
rng = np.random.default_rng(0)

## Parte A — O teste A/B clássico

### Passo 1 — Sample size
Taxa base de 5%, queremos detectar +10% relativo (5,0% → 5,5%), `α = 0,05`, power 80%.

In [ ]:
def tamanho_amostra(p1, mde_rel, alfa=0.05, poder=0.8):
    p2 = p1 * (1 + mde_rel)
    z = stats.norm.ppf(1 - alfa / 2) + stats.norm.ppf(poder)
    return int(np.ceil(z ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p2 - p1) ** 2))

n = tamanho_amostra(0.05, 0.10); print("n por grupo:", n)
for mde in (0.20, 0.10, 0.05, 0.02): print(f"MDE {mde:.0%} -> {tamanho_amostra(0.05, mde):>9,} por grupo")

Reduzir o MDE pela metade multiplica a amostra por quanto?

### Passo 2 — Analisar um teste
Simule um teste com efeito real de +10% e calcule: diferença, IC de 95% e p-value (teste z de duas proporções). Confira com `proportions_ztest`.

In [ ]:
a = rng.binomial(1, 0.050, n); b = rng.binomial(1, 0.055, n)
def analisar(a, b):
    pa, pb, na, nb = a.mean(), b.mean(), len(a), len(b)
    pool = (a.sum() + b.sum()) / (na + nb)
    z = (pb - pa) / np.sqrt(pool * (1 - pool) * (1 / na + 1 / nb))
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    ep = np.sqrt(pa * (1 - pa) / na + pb * (1 - pb) / nb)
    ic = (pb - pa - 1.96 * ep, pb - pa + 1.96 * ep)
    return {"dif": pb - pa, "ic95": ic, "p": p}

r = analisar(a, b); print(r)
assert abs(r["p"] - proportions_ztest([b.sum(), a.sum()], [len(b), len(a)])[1]) < 1e-9

### Passo 3 — Power, por simulação
Repita o experimento 2.000 vezes com efeito real e conte a fração de vezes em que `p < 0,05`. Depois, repita **sem** efeito: a fração é a taxa de false positives.

In [ ]:
def taxa_de_rejeicao(p_a, p_b, n, reps=2000):
    xa = rng.binomial(n, p_a, reps); xb = rng.binomial(n, p_b, reps)
    pool = (xa + xb) / (2 * n); z = (xb - xa) / n / np.sqrt(pool * (1 - pool) * 2 / n)
    return float(np.mean(2 * (1 - stats.norm.cdf(np.abs(z))) < 0.05))

print(f"poder (efeito real)      : {taxa_de_rejeicao(0.05, 0.055, n):.3f}  (planejado: 0.80)")
print(f"falsos positivos (A/A)   : {taxa_de_rejeicao(0.05, 0.050, n):.3f}  (planejado: 0.05)")
print(f"poder com 1/4 da amostra : {taxa_de_rejeicao(0.05, 0.055, n // 4):.3f}")

### Passo 4 — O custo de peeking
Teste A/A (sem efeito nenhum) com 20 dias. A cada dia calcula-se o p-value acumulado; para-se no primeiro `p < 0,05`. Qual a taxa de false positives?

In [ ]:
def espiar(dias=20, n_dia=1500, reps=1000):
    falsos = 0
    for _ in range(reps):
        xa = np.cumsum(rng.binomial(n_dia, 0.05, dias)); xb = np.cumsum(rng.binomial(n_dia, 0.05, dias)); N = n_dia * np.arange(1, dias + 1)
        pool = (xa + xb) / (2 * N); z = (xb - xa) / N / np.sqrt(pool * (1 - pool) * 2 / N)
        p = 2 * (1 - stats.norm.cdf(np.abs(z)))
        falsos += (p < 0.05).any()
    return falsos / reps

print(f"falsos positivos espiando todo dia: {espiar():.2f} (nominal: 0.05)")

### Passo 5 — SRM
O sorteio era 50/50. Chegaram 50.000 usuários no controle e 48.700 no tratamento. Use o qui-quadrado para decidir se isso é acaso.

In [ ]:
obs = [50_000, 48_700]
chi2, p_srm = stats.chisquare(obs, f_exp=[sum(obs) / 2] * 2)
print(f"p-valor do SRM: {p_srm:.2e} ->", "NÃO confie no teste: investigue a atribuição" if p_srm < 0.001 else "ok")

## Parte B — Quando os usuários interagem

### Passo 6 — Um produto com famílias
App de álbuns de fotos. Os usuários formam famílias de 2 a 6 pessoas que colaboram entre si. A feature de IA (títulos automáticos) tem:
- **efeito direto** `τ = 2,0` álbuns/mês em quem a recebe;
- **spillover** `γ = 1,2`, proporcional à fração de **outros** membros da família que foram tratados (eles convidam para álbuns).

Efeito de lançar para todos: `τ + γ = 3,2`.

In [ ]:
TAU, GAMA = 2.0, 1.2
def populacao(n_familias=4000, seed=1):
    r = np.random.default_rng(seed); tam = r.integers(2, 7, n_familias)
    fam = np.repeat(np.arange(n_familias), tam)
    efeito_familia = np.repeat(r.normal(0, 1.5, n_familias), tam)          # famílias mais ou menos ativas (gera ICC)
    return pd.DataFrame({"familia": fam, "base": 5 + efeito_familia + r.normal(0, 1.5, len(fam))})

def resultado(df, tratado, seed=2):
    d = df.assign(t=tratado)
    soma = d.groupby("familia")["t"].transform("sum"); n = d.groupby("familia")["t"].transform("count")
    d["frac_outros"] = (soma - d["t"]) / (n - 1)                             # fração dos OUTROS membros tratados
    d["y"] = d["base"] + TAU * d["t"] + GAMA * d["frac_outros"] + np.random.default_rng(seed).normal(0, 1, len(d))
    return d

pop = populacao(); print(len(pop), "usuários em", pop["familia"].nunique(), "famílias | efeito verdadeiro do lançamento total:", TAU + GAMA)

### Passo 7 — A/B por usuário
Sorteie cada usuário individualmente (50%). Estime o efeito pela diferença de médias.

In [ ]:
t_ind = rng.binomial(1, 0.5, len(pop))
d_ind = resultado(pop, t_ind)
est_ind = d_ind[d_ind.t == 1]["y"].mean() - d_ind[d_ind.t == 0]["y"].mean()
print(f"estimativa (randomização individual): {est_ind:.2f} | verdade: {TAU + GAMA}")
print("média de y no controle:", round(d_ind[d_ind.t == 0]["y"].mean(), 2), "| sem nenhum tratamento seria ≈ 5.00")

O controle foi contaminado: recebeu spillover dos familiares tratados. De quanto é o viés?

### Passo 8 — Randomização por cluster
Sorteie **famílias** inteiras. Todos os membros recebem a mesma atribuição.

In [ ]:
sorteio_fam = rng.binomial(1, 0.5, pop["familia"].nunique())
t_clu = sorteio_fam[pop["familia"].values]
d_clu = resultado(pop, t_clu)
est_clu = d_clu[d_clu.t == 1]["y"].mean() - d_clu[d_clu.t == 0]["y"].mean()
print(f"estimativa (randomização por cluster): {est_clu:.2f} | verdade: {TAU + GAMA}")

### Passo 9 — O erro-padrão correto
Na randomização por cluster, usuários da mesma família são correlacionados. Compare o erro-padrão ingênuo (OLS comum) com o erro-padrão robusto por cluster.

In [ ]:
ingenuo = smf.ols("y ~ t", d_clu).fit()
robusto = smf.ols("y ~ t", d_clu).fit(cov_type="cluster", cov_kwds={"groups": d_clu["familia"]})
print(f"EP ingênuo: {ingenuo.bse['t']:.4f} | EP por cluster: {robusto.bse['t']:.4f} | razão: {robusto.bse['t'] / ingenuo.bse['t']:.2f}")
print("IC95% correto:", robusto.conf_int().loc["t"].round(2).tolist())

### Passo 10 — ICC e efeito de desenho
Estime a correlação intraclasse no grupo de controle e o efeito de desenho `1 + (m − 1) · ICC`.

In [ ]:
c = d_clu[d_clu.t == 0]
var_entre = c.groupby("familia")["y"].mean().var(); var_dentro = c.groupby("familia")["y"].var().mean()
m = c.groupby("familia").size().mean()
icc = (var_entre - var_dentro / m) / (var_entre - var_dentro / m + var_dentro)
deff = 1 + (m - 1) * icc
print(f"ICC ≈ {icc:.2f} | tamanho médio da família {m:.1f} | efeito de desenho {deff:.2f} | amostra efetiva: {len(d_clu) / deff:,.0f} de {len(d_clu):,}")

### Passo 11 — Viés × variância, repetindo o experimento
Repita os dois desenhos 300 vezes. Compare a média das estimativas (viés) e o desvio-padrão (variância).

In [ ]:
def uma_vez(desenho, seed):
    r = np.random.default_rng(seed)
    t = r.binomial(1, 0.5, len(pop)) if desenho == "individual" else r.binomial(1, 0.5, pop["familia"].nunique())[pop["familia"].values]
    d = resultado(pop, t, seed=seed + 10_000); return d[d.t == 1]["y"].mean() - d[d.t == 0]["y"].mean()

sim = {des: np.array([uma_vez(des, s) for s in range(300)]) for des in ("individual", "cluster")}
for des, v in sim.items(): print(f"{des:10s} | média {v.mean():.2f} | desvio {v.std():.3f} | viés {v.mean() - (TAU + GAMA):+.2f}")
plt.hist(sim["individual"], bins=30, alpha=.6, label="individual"); plt.hist(sim["cluster"], bins=30, alpha=.6, label="cluster")
plt.axvline(TAU + GAMA, c="k", ls="--", label="verdade"); plt.legend(); plt.xlabel("efeito estimado"); plt.show()

**Reflexão:** para a pergunta "qual o efeito de lançar para todos?", o desenho individual é preciso e errado; o por cluster é ruidoso e certo. Repare que a média do individual fica perto de `τ = 2,0`: ele mede bem o **efeito direto** e perde o spillover `γ`. Em que situação essa seria justamente a pergunta de interesse?

## Teste Prático de Entrevista — Detector de Sample Ratio Mismatch (SRM)
Um entrevistador pede: *"Antes de analisar conversão de um Teste A/B, implemente `checar_srm(n_ctrl, n_treat, prop_treat=0.5)` calculando a estatística qui-quadrado de Pearson `(O_c - E_c)^2/E_c + (O_t - E_t)^2/E_t` e retornando `(chi2, srm_detectado)` com limiar crítico `10.828` (`p < 0.001`)."* 

In [ ]:
def checar_srm(n_ctrl, n_treat, prop_treat=0.5):
    """Retorna (chi2, srm_detectado) para split 50/50 ou arbitrário."""
    total = n_ctrl + n_treat
    e_treat = total * prop_treat
    e_ctrl = total * (1.0 - prop_treat)
    chi2 = ((n_ctrl - e_ctrl) ** 2) / e_ctrl + ((n_treat - e_treat) ** 2) / e_treat
    return chi2, chi2 > 10.828


chi_ok, srm_ok = checar_srm(10020, 9980, 0.5)
assert srm_ok is False and chi_ok < 1.0
chi_bad, srm_bad = checar_srm(10500, 9500, 0.5)
assert srm_bad is True and chi_bad > 10.828
print(f"OK! Split saudável chi2={chi_ok:.3f} | SRM detectado chi2={chi_bad:.1f}")